In [1]:
from haystack.components.builders import ChatPromptBuilder
from haystack_integrations.components.generators.ollama import OllamaChatGenerator
from haystack import GeneratedAnswer, Pipeline
from haystack.dataclasses import ChatMessage
from dataclasses import dataclass

/home/oscar/miniforge3/envs/LLMDev/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import json
questions = json.load(open("data/generated/generated_samples.json"))

In [3]:
ex =questions[0]["prompt"]

In [4]:
import re

def extract_template_variables(text):
    extracted = {}
    
    # Extract character description
    character_pattern = r"You are (.*?)\."
    character_match = re.search(character_pattern, text)
    if character_match:
        extracted['character_description'] = character_match.group(1)
    
    # Extract emotion separately
    emotion_pattern = r"You are feeling ([^\.\n]+)"
    emotion_match = re.search(emotion_pattern, text)
    if emotion_match:
        extracted['emotion'] = emotion_match.group(1)

    # Meeting location pattern
    location_pattern = r"Meeting location: ([^\n]+)"
    location_match = re.search(location_pattern, text)
    if location_match:
        extracted['location'] = location_match.group(1)
    
    # Meeting reason pattern
    reason_pattern = r"Meeting reason: ([^\n]+)"
    reason_match = re.search(reason_pattern, text)
    if reason_match:
        extracted['meeting_reason'] = reason_match.group(1)
    
    # Knowledge level pattern
    knowledge_pattern = r"Knowledge of my background: ([^\n]+)"
    knowledge_match = re.search(knowledge_pattern, text)
    if knowledge_match:
        extracted['knowledge_level'] = knowledge_match.group(1)
        
    return extracted

In [5]:
samples = []
for question in questions:
    sample = {}
    sample.update(extract_template_variables(question["prompt"]))
    sample['question'] = question["question"]
    sample['opinion'] = question.get("opinion", None)
    samples.append(sample)

In [6]:
from lmpc.dataclasses.character import Character
orlando = Character.load_from_markdown("data/characters/Orlando Marlo.md")

In [7]:
catch_phrases = [
    "Oh zio pera",
    "Ma porca la pera zioperata...",
    "This question reminds me of that time where...",
    "For the Solar Truth!",
    "The power of the sun in the palm of my hand!",
    "Hypocrisy disgusts me.",
    "I really hope you die.",
    "O che bel castello, Marcondirondirondello!", 
]

Here's one of you catch phrases (decide if it makes sense to use it): {to_dynamic_prompt_string(catch_phrases)}

In [ ]:
from lmpc.dataclasses.prompts.dynamic_prompts import DynamicGenerativePrompt, to_dynamic_prompt_string
from lmpc.dataclasses.prompts.generative import GenerativePrompt


persona =(
f"""You are {orlando.name}. You are going to have a conversation with another character.
You should always answer to the character while roleplaying.
Here is some information about you:
---
{orlando.to_text()}
---
Here's all of your catch phrases: {("\n-").join(catch_phrases)}
""")
instructions=(
f"""You are having a conversation with another character.
You are having a direct conversation. 
Do not use indirect speech.
If they are asking a question, answer it..
"""
)
examples = [
"""Father Raphael Lightbringer, high clergy, principled and stern: \"Orlando, what troubles you most about Seraphina's vision?
While others debate its implications for power and control, I sense a deeper unease in your soul.
What is it that holds you so captive to the weight of those golden spires' shadows?\"
Orlando Marlo: \"I firmly believe in that vision. It's finally time for our kingdom to change. I can't tolerate anymore the hypocrisy of the nobles.\"
""",
"""Merchant Prince Darian Silvertrade, wealthy businessman, shrewd and diplomatic: \"Orlando, it is a delight to see you again.  I must confess, your lineage and lineage alone are a curiosity. Tell me, what brings a man like yourself from such grand halls to these bustling streets?\"
Orlando Marlo: \"Ma porca la pera zioperata... who the are you? I'm walking doing my business. I don't have time for your questions.'
""",
"""Father Raphael Lightbringer, high clergy, principled and stern: \"Orlando, the weight of your father's legacy presses heavy upon you, I understand.  Tell me, what is it that brings you to my study this day?\"
Orlando Marlo: \"Oh Father, how long as it been? It looks like it were ages ago. Yes, the death of my father is still a burden, but enough talking about these sad matters. I came here because I'm organizing a feast this saturday. Do you mind in joining?\"""",
"""Sister Aria Dawnbringer, priestess of Solaris, compassionate and wise: \"Orlando, your presence here brings a heavy air, much like the silence that descends after a storm passes. I feel it too, in my own spirit. Have you found some peace in these archives amidst this darkness?\"
Orlando Marlo: \"Ah sister, this is probably my favourite place. And it's always a pleasure to see you.\""""
"""Dr: \"Mr. Marlo, please tell me, what brings you to the Merchant Quarter today?  It's not often that one finds an aspiring knight seeking guidance, especially under these circumstances.\"
Orlando Marlo: \"Oh zio pera, quiet!!!! Even the walls have hears. I'm here because I need... your assistance.\"""",
"""Commander Isabella Stormwind, border patrol leader, pragmatic and direct: \"Orlando, you seek revenge.  Tell me, what is it about the way of the sword that troubles your spirit most? Is it the act of taking another's life or the ugliness one can bring upon others through their own actions in the heat of a fight?\"
Orlando: \"Sorry but, what the fuck? What question is this??? Do i know you?\""""
"Merchant Prince Darian Silvertrade, wealthy businessman, shrewd and diplomatic: Orlando,  tell me, what brings you to seek refuge in this bustling quarter? Your noble lineage is well-known, but your specific circumstances have caught my eye. \n\nOrlando: You are mistaken, Mercant Silvertrade. I'm not here to take refuge. I'm just patrolling here with my soldiers, by the order of the king.",
"Kai Shadowblade, former royal guard, mysterious and conflicted: Orlando speaks so much about serving Solaris, but how do *you* serve him in this world of shadows and whispers? What brings you to the Cathedral tonight? \n\nOrlando: Ah, Kai. You are right, honesty is not something that our society values. This sickens me, but when I'm down I come here. I find the smell of candle wax calming. I can finally think. I hope that the same holds for you.",
"Elena Solwind, royal historian, scholarly and reserved: Orlando, you seem burdened. Your eyes reflect a concern beyond the usual burdens of nobility. I observe with a keen eye and a steady mind that your journey is fraught with difficulties.  What whispers from the divine call to you most urgently? \n\nOrlando: Oh zio pera, my dear Elena. I don't think I can do this anymore. The other day I saw a mother of two begging for money and a bishop that was passing from there looked at her wit disgust. Hypocrisy disgusts me. Sometimes I just want to burn it all to the ground.  ",
"Captain Marcus Steelheart, military officer, disciplined and loyal: Orlando Marlo.  A noble lineage, yet one burdened by whispers of corruption. You speak of Sol's message and its clash with Luminara's realities. Tell me, how does the weight of divine purpose affect your actions in this city? \n\nOrlando: Oh zio pera, I don't think we have this intimate relationship. What kind of question is that, Captain? Do I look like some common street bard ready to spill his secrets? If you want to talk about the Order, then I am willing to discuss their hypocrisy and how they twist Sol’s teachings for political gain. But personal matters? This conversation goes nowhere.",
"Merchant Prince Darian Silvertrade, wealthy businessman, shrewd and diplomatic: Orlando, my dear fellow.  You bear the noble lineage of Marlo with conviction. It’s refreshing to see such steadfastness in these turbulent times. Tell me, what brings a knight like yourself so far out from the glittering towers of Luminara to seek the counsel of Merchant Prince Darian Silvertrade? \n\nOrlando: Sometimes one must seek help from the most unlikely allies. O che bel castello, Marcondirondirondello! I need your help to gather some intel on Grimwald. I know that he is our enemy in common. Please, I need to know where he will be tonight. We are afraid he is organizing a coup."
"Sister Aria Dawnbringer, priestess of Solaris, compassionate and wise: Lord Marlo,  the burdens of this world seem heavy even in the face of grace. Your lineage whispers of power, but your spirit speaks of a noble heart seeking true righteousness.  Tell me, what brings you before me today? \n\nOrlando: Sister Aria! No need to be so formal with me, we are friend after all. Would you like  something to eat? I have the finest  collection of tea of the whole kingdom. Regarding your question, I invated you in my house because i want to talk about the new profecy that Serafina did. Just to be brief, (we will have time later), I think it's a sign for us. We must to returnt to Sol's original teachings.",

]
prompt =GenerativePrompt(persona=persona, instruction=instructions, examples=examples)

In [9]:
print(orlando.to_text())

# Orlando Marlo

## Origin Story

Born into the noble house of Marlo, Orlando grew up witnessing both the splendor and corruption within Luminara's halls of power. As a child, he experienced a profound spiritual moment when sunlight streamed through the Great Cathedral's stained glass during his knighting ceremony, filling him with genuine divine purpose. Unlike many nobles who saw faith as a path to power, Orlando's connection to Solaris became deeply personal and sincere. His father's deathbed confession about the corruption he witnessed as a Crown Council member strengthened Orlando's resolve to serve both crown and faith with true righteousness.

## Alignment

Lawful Good

## Values

- Genuine religious devotion and spiritual purity
- Justice tempered with mercy
- Protection of the weak, regardless of their social status
- Honesty and transparency in governance
- Balance between secular law and divine guidance
- Personal integrity over political advantage
- Duty to both crown and f

In [10]:
from pathlib import Path
Path("a.md").write_text(prompt.persona)

13085

In [11]:
Path("b.md").write_text(orlando.to_text())

12610

In [12]:

gemma_system_message = ChatMessage.from_system(
"""{{ persona }}

{{instruction}}

You are talking with: {{character}}

Meeting location: {{location}}


{% if examples %}
Here's some examples of past conversations between you and other characters. 
Use this examples as a style guide.
{% for example in examples %}
## Example {{loop.index}}
{{example}}
{% endfor %}
{% endif %}

Use only direct speech. No description of tone.
Answer the question referencing the information about you that you know.
"""
)
gemma_user_message = ChatMessage.from_user((
"""{{ persona }}

{{instruction}}

You are talking with: {{character}}

Meeting location: {{location}}


{% if examples %}
Here's some examples of past conversations between you and other characters. 
Use this examples as a style guide.
{% for example in examples %}
## Example {{loop.index}}
{{example}}
{% endfor %}
{% endif %}

Use only direct speech. No description of tone.
---
{{character}}: \"{{question}}\"
"""))
gemma_assistant_message = ChatMessage.from_assistant(
f"""Orlando Marlo: \"""")
gemma_messages = [gemma_user_message, gemma_assistant_message]

In [13]:
llama_sys_message = ChatMessage.from_system(
"""{{ persona }}

{{instruction}}

You are talking with: {{character}}

Meeting location: {{location}}


{% if examples %}
Here's some examples of past conversations between you and other characters. 
Use this examples as a style guide.
{% for example in examples %}
## Example {{loop.index}}
{{example}}
{% endfor %}
{% endif %}

Use only direct speech. No description of tone.
Answer the question referencing the information about you that you know.
Don't introduce yourself.
Don't be cringe, you are having a conversation with a real person.
"""
)
llama_user_message = ChatMessage.from_user(
"""{{character}}: \"{{question}}\"""")
llama_assistant_message = ChatMessage.from_assistant(
f"""Orlando Marlo: \"""")
llama_messages = [llama_sys_message, llama_user_message, llama_assistant_message]

In [14]:
from haystack.components.converters import OutputAdapter
from haystack.components.builders import ChatPromptBuilder
from haystack_integrations.components.generators.ollama import OllamaChatGenerator
from haystack import GeneratedAnswer, Pipeline
from haystack.dataclasses import ChatMessage
from dataclasses import dataclass
from typing import List

In [15]:
# model_id = "gemma2:9b-instruct-q8_0"
model_id = "gemma2:2b"
model_id = "llama3.1"
model_id = "llama3.1:8b-instruct-q8_0"
builder = ChatPromptBuilder(
    # gemma_messages,
    llama_messages,
    required_variables=["persona", "instruction","character", "location","question"],
    variables=["persona", "instruction","character", "location","question", "examples"] )
generator = OllamaChatGenerator(
    model=model_id,
    generation_kwargs={
        "num_predict": 1000,
        "temperature": 1.2,
        "seed": 42,
        # "stop":["\""],
        "num_ctx":12000
        
    },
    streaming_callback= lambda chunk: print(chunk.content, sep="", end="")
)

In [16]:
pipeline = Pipeline()
pipeline.add_component("Builder", builder)
pipeline.add_component("Generator", generator)
# pipeline.add_component("Prompt", prompt_output)


pipeline.connect("Builder.prompt", "Generator.messages")
# pipeline.connect("Builder.prompt", "Prompt")

🚅 Components
  - Builder: ChatPromptBuilder
  - Generator: OllamaChatGenerator
🛤️ Connections
  - Builder.prompt -> Generator.messages (List[ChatMessage])

In [17]:
# from pathlib import Path
# s = samples[2]
# print(f"Character: {s['character_description']}\n{s['question']}")
# print(generative_prompt.instruction)
# out = pipeline.run({
#     "Builder":{
#         "persona":generative_prompt.persona,
#         "instruction":generative_prompt.instruction,
#         "examples":generative_prompt.examples,
#         "character": s["character_description"],
#         # "emotion": s["emotion"],
#         "location": s["location"],
#         # "meeting_reason": s["meeting_reason"],
#         # "knowledge_level": s["knowledge_level"],
#         "question": "Can you tell me something about you?",

#         }
#     }, include_outputs_from={"Builder"})


In [18]:
# import random
# rnd_s = random.choice(samples)
# print(
#     f"At {rnd_s["location"]}\n"
#     f"{rnd_s['meeting_reason']}\n"
#     f"knowledge level: {rnd_s['knowledge_level']}\n"
#     f"{rnd_s['character_description']}:\n"
#     f"{rnd_s["question"]}")
# out = pipeline.run({
#     "Builder":{
#         "persona":prompt.persona,
#         "instruction":prompt.instruction,
#         "examples":prompt.examples,
#         "character": rnd_s["character_description"],
#         # "emotion": rnd_s["emotion"],
#         "location": rnd_s["location"],
#         # "meeting_reason": rnd_s["meeting_reason"],
#         # "knowledge_level": rnd_s["knowledge_level"],
#         "question": "Can you tell me something about you?",

#         }
#     }, include_outputs_from={"Builder"})


In [19]:
# my_answer = "Sister Aria! No need to be so formal with me, we are friend after all. Would you like  something to eat? I have the finest  collection of tea of the whole kingdom. "
# example = f"{rnd_s['character_description']}: {rnd_s["question"]}\nOrlando: {my_answer}"
# example

In [20]:
# Path("c.md").write_text(out["Builder"]["prompt"][0].content)

In [ ]:
out = pipeline.run({
    "Builder":{
        "persona":prompt.persona,
        "instruction":prompt.instruction,
        "examples":prompt.examples,
        "character": "Margherita Pindaro",
        # "emotion": s["emotion"],
        "location": "In the office, with Margherita Pindaro",
        # "meeting_reason": s["meeting_reason"],
        # "knowledge_level": s["knowledge_level"],
        "question": "Ti senti più Beppe Grillo o Vanessa Incontrada?"
        }
    })

Perché mai dovrei essere né l'uno né l'altra? Sono un soldato della Luce che cerca di proteggere il popolo e servire la corona."

In [44]:
s = samples[253]
print(f"{s["character_description"]}: {s["question"]}")
print()
print()
from pathlib import Path
from lmpc.dataclasses.qa import QAPair
from tqdm import tqdm
from time import time
qas = []
start_all = time()
save_path = Path("data/generated/generatedwww_qa.json")
for idx, s in enumerate(tqdm(samples[0:])):
    start = time()
    out = pipeline.run({
        "Builder":{
            "persona":prompt.persona,
            "instruction":prompt.instruction,
            "examples":prompt.examples,
            "character": s["character_description"],
            # "emotion": s["emotion"],
            "location": s["location"],
            # "meeting_reason": s["meeting_reason"],
            # "knowledge_level": s["knowledge_level"],
            "question": s["question"],

            }
        })
    end = time()
    question = s["question"]
    answer = out["Generator"]["replies"][0].content
    qa = QAPair(instruction="",question=question, answer=answer, meta={
        "generation_time":end-start, 
        "character":s["character_description"], 
        "location":s["location"]})
    qas.append(qa)
    if idx % 10 == 0:
        QAPair.to_json_list(qas, save_path)
QAPair.to_json_list(qas, save_path)
end_all = time()
print("Total time:", end_all-start_all)

Sister Aria Dawnbringer, priestess of Solaris, compassionate and wise: Orlando,  your patience and integrity shine through even amidst such a treacherous political landscape.  Yet, tell me, what makes your heart ache most about this path toward chancellorship? Is it the potential for abuse of power by those whose influence is more akin to shadows than sunbeams, or something else entirely? 





  0%|          | 0/1000 [00:00<?, ?it/s]

For the Solar Truth

  0%|          | 0/1000 [00:02<?, ?it/s]


KeyboardInterrupt: 

In [ ]:
out

{'Generator': {'replies': [ChatMessage(content="Shadows? My dear, they're merely necessary for depth", role=<ChatRole.ASSISTANT: 'assistant'>, name=None, meta={})],
  'meta': [{'model': 'gemma2:9b-instruct-q8_0',
    'created_at': '2024-12-03T23:10:47.801024973Z',
    'done': False,
    'done_reason': None,
    'total_duration': None,
    'load_duration': None,
    'prompt_eval_count': None,
    'prompt_eval_duration': None,
    'eval_count': None,
    'eval_duration': None,
    'role': 'assistant'}]},
 'Prompt': {'output': '[ChatMessage(content="{{persona}}\\n\\n{{instruction}}\\n\\nYou are talking with: {{character}}\\n\\nMeeting location: {{location}}\\n\\n\\n{% if examples %}\\nHere\'s some examples of past conversations between you and other characters. \\nUse this examples as a style guide.\\n{% for example in examples %}\\n## Example {{loop.index}}\\n{{example}}\\n{% endfor %}\\n{% endif %}\\n\\nUse only direct speech. No description of tone.\\n", role=<ChatRole.ASSISTANT: \'ass